# 05 Velocity-Segmented Calibration H05

H01-H03 closed the activation-window family as a useful segment signal but not a champion path. H04 closed generic sell-price momentum as weak global evidence after an extended backtest.

The next hypothesis targets the repeated failure mode: forecast metrics and business decision metrics keep moving in different directions. This notebook sets up H05 before the ShelfOps run so the experiment has a clear retail rationale and acceptance criteria.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != 'shelfops_project' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

H03_REPORT = PROJECT_ROOT / 'backend' / 'reports' / 'experiments' / '0e22686c-b05a-40a0-977c-8977de1f373e.report.json'
H04_EXTENDED_REPORT = PROJECT_ROOT / 'backend' / 'reports' / 'experiments' / '514960f9-2496-4177-8aad-fb0418f89dfe.report.json'
EDA_JSON = PROJECT_ROOT / 'backend' / 'reports' / 'experiments' / 'manual_ds' / '00_m5_global_eda' / 'm5_global_eda.json'

def load_json(path):
    with path.open() as f:
        return json.load(f)

h03 = load_json(H03_REPORT) if H03_REPORT.exists() else None
h04 = load_json(H04_EXTENDED_REPORT) if H04_EXTENDED_REPORT.exists() else None
eda = load_json(EDA_JSON) if EDA_JSON.exists() else None

print('H03 loaded:', bool(h03))
print('H04 extended loaded:', bool(h04))
print('EDA loaded:', bool(eda))

## Senior DS Read Before H05

I am not moving to H05 because I want to keep trying random model changes. I am moving to H05 because the prior runs show a repeated business tradeoff:

- H01/H02 reduced lost-sales pressure but worsened overstock or bias.
- H03 improved WAPE, MASE, bias, coverage, and overstock, but worsened stockout and lost-sales risk.
- H04 had plausible retail rationale, but temporal validation showed weak and unstable global benefit.

That points to segmentation/calibration rather than another broad feature. M5 demand is sparse and velocity-skewed, so the next test should ask whether sparse and fast-moving items need different calibration behavior.

In [ ]:
if eda:
    demand_shape = eda.get('summary', {})
    intermittent = eda.get('intermittent_demand_summary', {})
    velocity = eda.get('velocity_summary', [])

    rows = []
    for row in intermittent.get('taxonomy', []):
        rows.append({
            'slice': row.get('demand_classification'),
            'series': row.get('series'),
            'unit_share': row.get('unit_share'),
            'avg_nonzero_rate': row.get('avg_nonzero_rate'),
            'median_adi': row.get('median_adi'),
            'median_cv2': row.get('median_cv2'),
        })
    display(pd.DataFrame(rows))

    display(pd.DataFrame([{
        'zero_sales_row_rate': demand_shape.get('zero_sales_rate'),
        'top_10pct_series_unit_share': demand_shape.get('top_10pct_series_unit_share'),
    }]))

    display(pd.DataFrame(velocity)[['velocity_segment', 'series', 'total_units', 'unit_share', 'avg_nonzero_rate']])

## H05 Hypothesis

M5 demand is velocity-segmented and heavily intermittent. Adding intermittency features with category/velocity bias calibration should improve slow/intermittent forecast behavior and reduce replenishment cost without pushing stockout risk or buyer workload higher.

Executable ShelfOps spec: `m5_velocity_segmented_bias_v1`.

This is intentionally not another activation experiment and not another price-momentum experiment.

## Success Criteria Before Running

Primary model read:

- Global WAPE and MASE improve or remain flat.
- Slow/intermittent/lumpy slices improve without sacrificing fast/high-volume slices.
- Bias and interval coverage do not materially worsen.

Decision read:

- Combined cost proxy does not regress.
- Overstock dollars improve or remain flat.
- Lost-sales quantity and stockout opportunity cost do not regress materially.
- Service level remains within gate.
- PO count should not increase enough to imply buyer workload regression.

If this fails, the next move is likely a decision-policy hypothesis, not another forecast-feature hypothesis.

## ShelfOps Run Controls

| Control | Value |
|---|---|
| Model family | Forecasting |
| Source | Manual DS |
| Spec | `m5_velocity_segmented_bias_v1` |
| Experiment name | `m5_velocity_segmented_calibration_h05` |
| Validation mode | Extended Backtest |
| Holdout days | 28 |
| Calibration days | 28 |
| Rolling windows | 6 |
| Rolling window days | 28 |
| Rolling stride days | 28 |
| Max rows | 250000 |
| Max series | 180 |

## Post-Run Cells To Fill

After running H05 through ShelfOps, load the report JSON here and compare:

- champion versus H05 primary holdout metrics,
- decision replay metrics,
- rolling validation window pass/fail behavior,
- segment metrics for fast, high-volume, medium, slow, intermittent, and lumpy slices where available,
- promotion gate failures,
- final senior DS decision.